In [ ]:
import os
import re
import zipfile
import io
import csv
from PIL import Image
from tqdm import tqdm

In [ ]:
def extract_atlas_sprites(image_id: str, atlas_name: str):
    """
    Parses VT Lua atlas file and extracts individual sprites from the source
    :param image_id: The name of the image file (e.g., "EED797CC825CF42A").
    :param atlas_name: The name of the atlas type (e.g., "forge").
    """
    image_path = f"/content/{image_id}.png"
    atlas_path = f"/content/gui_{atlas_name}_atlas.lua"
    out_zip_path = f"/content/gui_{atlas_name}_atlas.zip"
    work_dir = f"/content/gui_{atlas_name}_atlas_temp"

    os.makedirs(work_dir, exist_ok=True)

    if not os.path.isfile(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    if not os.path.isfile(atlas_path):
        raise FileNotFoundError(f"Atlas Lua not found: {atlas_path}")

    img = Image.open(image_path).convert("RGBA")
    W, H = img.size
    print(f"Loaded image {image_path} size={W}x{H}")

    with open(atlas_path, "r", encoding="utf-8", errors="ignore") as f:
        lua_text = f.read()

    start_idx = lua_text.find(f"{atlas_name}_atlas")
    if start_idx == -1:
        raise ValueError(f"No header found for {atlas_name}_atlas (start_idx)")

    brace_open = lua_text.find("{", start_idx)
    if brace_open == -1:
        raise ValueError("Found header without {")

    depth = 0
    end_idx = None
    for i in range(brace_open, len(lua_text)):
        ch = lua_text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                end_idx = i
                break

    if end_idx is None:
        raise ValueError("} absent")

    atlas_body = lua_text[brace_open+1:end_idx]

    entries = []
    pos = 0
    while pos < len(atlas_body):
        m = re.search(r'\S', atlas_body[pos:])
        if not m:
            break
        pos += m.start()

        # try to match identifier key
        key_match = re.match(r"""
            (?:
              (?P<ident>[A-Za-z0-9_\-]+)
            |
              \["(?P<qident>[^"]+)"\]
            )
            \s*=\s*{""", atlas_body[pos:], flags=re.X)

        if not key_match:
            pos += 1
            continue

        key = key_match.group("ident") or key_match.group("qident")

        # find { for entry
        entry_open = pos + key_match.end() - 1

        # find } for entry
        depth = 0
        entry_close = None
        for i in range(entry_open, len(atlas_body)):
            ch = atlas_body[i]
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    entry_close = i
                    break

        if entry_close is None:
            print(f"Warning: no closing for '{key}', skip")
            pos = entry_open + 1
            continue

        entry_text = atlas_body[entry_open+1:entry_close]
        entries.append((key, entry_text))
        pos = entry_close + 1

    print(f"Found {len(entries)} entries")

    size_re = re.compile(r'size\s*=\s*{\s*(\d+)\s*,\s*(\d+)\s*,?\s*}', flags=re.I)
    uv_re = re.compile(r'uv(00|11)\s*=\s*{\s*([0-9]*\.?[0-9]+)\s*,\s*([0-9]*\.?[0-9]+)\s*,?\s*}', flags=re.I)

    def clamp(v, a, b):
        return max(a, min(b, v))

    def sanitize_filename(name):
        return re.sub(r'[^A-Za-z0-9_\-\.]', '_', name)

    manifest_rows = []

    for key, body in tqdm(entries):
        # find size
        sm = size_re.search(body)
        uvs = {}
        for um in uv_re.finditer(body):
            idx = um.group(1)
            u = float(um.group(2))
            v = float(um.group(3))
            uvs[idx] = (u, v)

        if '00' not in uvs or '11' not in uvs:
            print(f"Skip '{key}': no uv00 or uv11")
            continue
        if not sm:
            print(f"Skip '{key}': no size")
            continue

        w_out = int(sm.group(1))
        h_out = int(sm.group(2))
        (u0, v0) = uvs['00']
        (u1, v1) = uvs['11']

        # convert to pixel coords
        x0 = int(round(u0 * W) - 1)
        y0 = int(round(v0 * H) - 1)
        x1 = int(round(u1 * W))
        y1 = int(round(v1 * H))

        x0 = clamp(x0, 0, W)
        x1 = clamp(x1, 0, W)
        y0 = clamp(y0, 0, H)
        y1 = clamp(y1, 0, H)

        # swap if needed
        if x1 < x0:
            x0, x1 = x1, x0
        if y1 < y0:
            y0, y1 = y1, y0

        if x1 == x0 or y1 == y0:
            print(f"Skip '{key}': ({x0},{y0})-({x1},{y1})")
            continue

        cropped = img.crop((x0, y0, x1, y1))

        safe = sanitize_filename(key)
        out_png = os.path.join(work_dir, f"{safe}.png")
        cropped.save(out_png, format="PNG")

        manifest_rows.append({
            "name": key,
            "file": f"{safe}.png",
            "pixel_rect": f"{x0},{y0},{x1},{y1}",
            "output_size": f"{w_out}x{h_out}"
        })

    print(f"\n{len(manifest_rows)} sprites saved to {work_dir}.")

    with zipfile.ZipFile(out_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for r in manifest_rows:
            full = os.path.join(work_dir, r["file"])
            zf.write(full, arcname=r["file"])

    print("Created zip:", out_zip_path)

In [ ]:
# Example usage:
extract_atlas_sprites("BA5FEA2BF65D15BE", "items")

Loaded image /content/BA5FEA2BF65D15BE.png size=4096x4096
Found 1528 entries


100%|██████████| 1528/1528 [00:05<00:00, 278.05it/s]



1528 sprites saved to /content/gui_items_atlas_temp.
Created zip: /content/gui_items_atlas.zip
